## fMRI analysis for an example subject
Loads repeated fMRI responses for a single example subject, computes direct and indirect trajectory-curvature estimates across ROIs and temporal resolutions, and generates voxelwise cortical maps of implied curvature. This notebook supports Fig. 2a, Fig. 3, and Extended Data Fig. 3.

In [ ]:
from pathlib import Path
import sys

# Find repository root
repo_root = Path.cwd()

while not (repo_root / "src").exists():
    if repo_root == repo_root.parent:
        raise RuntimeError("Could not find repository root containing src/")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

%load_ext autoreload
%autoreload 1
%aimport src.fmri_core

# Imports
import os
import numpy as np
import cortex
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.linalg
import statsmodels.api as sm

%config InlineBackend.figure_format = "retina"

np.random.seed(0)

from src.fmri_core import (
    build_roi_voxs,
    load_response_trials,
    avg_pop_curvature,
    compute_coherence_spectrum,
    lowpass_and_resample,
    get_xfm,
    zs,
)

#### Load language data - original sampling (Fig 2a)

In [ ]:
subject = "S2"
story = "wheretheressmoke"

# Load data
response_trials = load_response_trials(subject, story)
print("response_trials:", response_trials.shape)  # (reps, time, voxels)

# Load ROI voxels
roi_voxs = build_roi_voxs(subject)
for roi, vox in roi_voxs.items():
    print(f"{roi:15s}: {len(vox):5d} voxels")

##### Direct curvature estimates (Fig 2a orange line)

In [ ]:
run_avg_resp = np.mean(response_trials, axis=0) # (nT, voxels) 
dir_roi_curv = {}
for roi in roi_voxs:
    roi_voxels = roi_voxs[roi]
    curvature = avg_pop_curvature(run_avg_resp[:, roi_voxels].T)  # calculate curvature
    dir_roi_curv[roi] = curvature
    print(f'{roi} curvature: {curvature:.2f}')  # print curvature value

roi_names = ["AC", "sPMv", "AG", "PFC"]
roi_curvatures = list(dir_roi_curv.values())
plt.figure(figsize=(4, 4))
plt.plot(
    roi_names,
    roi_curvatures,
    marker="o",
    linewidth=1.5
)
plt.xlabel("ROI")
plt.ylabel("Curvature (degrees)")
plt.title("Direct curvature estimates")
plt.ylim(90, 120)
plt.tight_layout()
plt.show()

##### AR(1) fit to run- and ROI-averaged responses (Fig 2a, black line)

In [ ]:
zs = lambda v: (v-v.mean(0))/(v.std(0) + 1e-8)
avg_reps = np.mean(response_trials, axis=0)
roi_ar1_coeff = {}
roi_ind_curv = {}

# Fit to voxel average
for roi in roi_voxs:
    roi_demean_resp = zs(np.mean(avg_reps[:,roi_voxs[roi]], axis=1)) # run average x voxel average
    ar1_model = sm.tsa.AutoReg(roi_demean_resp, lags = 1).fit()
    ar1_coeff = ar1_model.params[1]
    roi_ar1_coeff[roi] = ar1_coeff
    x = (ar1_coeff - 1) / 2
    x = np.clip(x, -1, 1)
    ind_curv = np.rad2deg(np.arccos(x))
    roi_ind_curv[roi] = ind_curv
    print(f'{roi} curvature: {roi_ind_curv[roi]:.2f}')

roi_names = ["AC", "sPMv", "AG", "PFC"]
roi_curvatures = list(roi_ind_curv.values())
plt.figure(figsize=(4,4))
plt.plot(
    roi_names,
    roi_curvatures,
    marker="o",
    linewidth=1.5
)
plt.xlabel("ROI")
plt.ylabel("Curvature (degrees)")
plt.title("Indirect curvature estimates")
plt.ylim(90, 120)
plt.tight_layout()
plt.show()



##### Estimate voxelwise AR(1) coefficients for the cortical curvature map (Fig 3)

In [ ]:
# data
zs = lambda v: (v-v.mean(0))/(v.std(0) + 1e-8)
avg_reps = np.mean(response_trials,0) 
# voxel_ar1 = np.empty(avg_reps.shape[1], dtype=float)
voxel_curv = np.full(avg_reps.shape[1], np.nan, dtype=float)
for voxel_idx in range(avg_reps.shape[1]):
    x = avg_reps[:, voxel_idx]

    # skip voxels with NaNs/infs or no variation
    if not np.all(np.isfinite(x)):
        continue
    if np.nanstd(x) < 1e-8:
        continue

    voxel_demean_resp = zs(x)
    ar1_model = sm.tsa.AutoReg(voxel_demean_resp, lags = 1).fit()
    # voxel_ar1[voxel_idx] = ar1_model.params[1]
    alpha = ar1_model.params[1]
    voxel_curv[voxel_idx] = np.degrees(np.arccos((alpha - 1)/2))

# alpha (compute coherence as repeatability)
freqs,_,_,_,_,coherence_lower = compute_coherence_spectrum(response_trials, fs=0.5, nfft=64)
print(coherence_lower.shape)
print(np.nanmin(coherence_lower), np.nanmax(coherence_lower))


In [ ]:
mean_coh = np.nanmean(coherence_lower, axis=0)

# Keep only reliable voxels
p = 80
finite = np.isfinite(mean_coh)
coh_thr = np.nanpercentile(mean_coh[finite], p)
coh_floor = 0.05

keep_mask = finite & (mean_coh >= coh_thr) & (mean_coh >= coh_floor)

# Mask unreliable voxels
masked_curv = np.array(voxel_curv, dtype=np.float32, copy=True)
masked_curv[~keep_mask] = np.nan

# Color scale based on retained voxels
kept_vals = masked_curv[np.isfinite(masked_curv)]
vmin, vmax = np.nanpercentile(kept_vals, [5, 95])

# Build pycortex volume
xfm = get_xfm(subject)

vol = cortex.Volume(
    masked_curv,
    subject,
    xfmname=xfm,
    cmap="hot",
    vmin=vmin,
    vmax=vmax
)

flatmap_kwargs = dict(
    recache=False,
    pixelwise=True,
    sampler="nearest",
    height=700,
    with_rois=True,
    with_curvature=True,
    curvature_threshold=True,
    linewidth=4,
    nanmean=True
)

fig = cortex.quickshow(
    vol,
    with_labels=False,
    colorbar_location=(0.8, 0.1, 0.3, 0.03),
    **flatmap_kwargs
)

In [ ]:
cortex.webshow(
    vol,
    with_rois=True,      # ROI boundaries
    with_labels=False,   # no ROI text labels
    linewidth=2
)